In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts', 'Dates']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives', 'JioSaavn']


In [3]:
from lib import allmusic
mio   = allmusic.MusicDBIO(verbose=False)
webio = allmusic.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AllMusic DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AllMusic


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
searchArtists      = mio.data.getSearchArtistData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artists:             {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AllMusic Search Results
   Global Master Search:      761013
   Global Master Search Data: 0
   Local Artists:             1461170
   Errors:                    669
   Search Artists:            1803604
   Known Summary IDs:         1455842


# Search For New Artists

In [ ]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [ ]:
useKnownData  = False
useMasterData = True

if useKnownData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].notna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  50696
#   Artist Names To Get:  42803
#   Artist Names To Get:  33587
#   Artist Names To Get:  21888
#   Artist Names To Get:  10630

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(60)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,list):
                for item in searchTermData:
                    if isinstance(item,dict):
                        artistID = item['id'][2:] if isinstance(item.get('id'),str) else None
                        if artistID is not None:
                            searchData[artistID] = {k: v for k,v in item.items() if k in ['name','ref']}
        df = DataFrame(searchData).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = allmusic.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF[searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} New Artists".format(searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    allmusic.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
    
#Found 1628828 Unique Results
#Found 1639245 Unique Results
#Found 1664572 Unique Results
#Found 1674908 Unique Results

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [18]:
useKnown = False

artistNames       = searchArtists
localArtistsDict  = localArtists.get()
artistNamesToGet  = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useKnown is True:
    pdbio = PanDBIO()
    pdbio.setData()
    matchedIDs = pdbio.getNotNaDBIDs(db)[db]
    artistNamesToGet = artistNamesToGet[artistNamesToGet.index.isin(matchedIDs)]
    del pdbio

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  429684
#   Artist Names To Get:  395684
#   Artist Names To Get:  375609
#   Artist Names To Get:  342434
#   Artist Names To Get:  310884

# AllMusic Search Results
#   Available Names:      1803604
#   Known Artist Names:   1514095
#   Artist Names To Get:  289509


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "9:50am")
tt = TermTime("today", "10:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["name"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting AllMusic ArtistIDs] Start    ==> Time Is 2022-06-25 10:45:06
========================= termTime(day=today,time=10:00pm) =========================
   ====> Terminate Time Set To 2022-06-25 22:00:00 <====
   ====> Will Terminate Process 11 Hours and 14 Minutes From Now
Getting Patrick McDaniel (0001482240)                     True
Getting Patrick McDonnell (0001256506)                    True
Getting Patrick McDonnell (0001882192)                    True
Getting Patrick McDanel (0002748096)                      True
Getting Jay Forbes (0001363365)                           True
Getting Joe Forbes (0001254979)                           True
Getting Jo Forbes (0002421514)                            True
Getting Ton Vingerhoets (0003832321)                      True
Getting Deep Piece (0000229633)                           True
Getting Kacie Sheik (0001718300)                          True
Getting Kacie Gilbert (0001397732)                        True
Getting Kacie Parker (

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Kevin Cooper (0000259427)                         True
Getting Kevin Cooper (0003298575)                         True
Getting Kevin Cooper (0003499771)                         True
Getting Kevin Kuper (0001618163)                          True
Getting Kevin Kupper (0004092197)                         True
Getting Gavin Cooper (0001183546)                         True
Getting Guy Audenaert (0002166740)                        True
Getting Stevie Saint (0000625932)                         True
Getting Sandi & Stevie (0001622876)                       True
Getting Steve Saint (0001019487)                          True
Getting Peter Gulas (0002346559)                          True
Getting Pedro Quiles (0000250978)                         True
Getting Georges Terme (0002995174)                        True
Getting Akkura (0003515387)                               True
Getting The 3 Ghouls (0000027141)                         True
Getting The Suicide Ghouls (0002714997)                

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Darren Wolff (0001346231)                         True
Getting Dan Weil (0003717996)                             True
Getting Dan Der Waal (0003114214)                         True
Getting Mike Nemirovsky (0002316014)                      True
Getting Joshua Patrick (0000482849)                       True
Getting Jay Lincoln (0001196610)                          True
Getting Joe Lincoln (0003788071)                          True
Getting Mika Kurvinen (0003484529)                        True
Getting Jerry Beach (0000276696)                          True
Getting Georges Bonneau (0001647044)                      True
Getting Jay Leslie (0000634106)                           True
Getting Jay Leslie (0001675337)                           True
Getting Leslie Joy (0000821442)                           True
Getting Leslie Joy (0001765523)                           True
Getting Joe Leslie (0001851903)                           True
Getting Nadia Teichmann (0003805224)                   

...

Getting Grady Kenevan (0001400821)                        True
Getting Simon Kenevan (0001532637)                        True
Getting Lee Canvana (0001765166)                          True
Getting Lena Cullen (0002089427)                          True
Getting Lena Cline (0003317054)                           True
Getting Lena Klein (0003437360)                           True
Getting Xel-Ha (0001583144)                               True
Getting Sioeli H. Tupou (0002385433)                      True
Getting H. Zell (0003602730)                              True
Getting George Mukabi (0001657888)                        True
Getting Tom Cat Norton (0003294648)                       True
Getting Splash 'N The Pants (0000747626)                  True
Getting Sid Luscious and the Pants (0002085433)           True
Getting Axel Pfeiffer (0003064452)                        True
Getting Angel Ice (0000035078)                            True
Getting Miguel Angel Iseas (0003911163)                

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Private Beat (0002081292)                         True
Getting Cybergang (0003947779)                            True
Getting Band One On (0001595881)                          True
Getting Rhizome Joue Ballif (0001460655)                  True
Getting Uprock Rhizome (0000525509)                       True
Getting Full Rhizome (0003624356)                         True
Getting Rhizome.S (0002913902)                            True
Getting SDEM (0003834007)                                 True
Getting Waldkauz (0003604721)                             True
Getting WooltyxX (0004213762)                             True
Getting Jody Wildgoose (0000981167)                       True
Getting Sebastian Weldycz (0002434830)                    True
Getting Joel Wildgoose (0003161266)                       True
Getting Svend Åge Buemann (0003220822)                    True
Getting Nani Castle (0003668196)                          True
Getting Lil' Bing (0000271526)                         

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Mollie Magee (0000483955)                         True
Getting Brian Magee (0000621554)                          True
Getting Joseph Magee (0000826288)                         True
Getting Robin MaGee (0001198130)                          True
Getting Mitch Magee (0001233912)                          True
Getting Steve Magee (0001244003)                          True
Getting LıssA (0003610041)                                True
Getting Betty Ann McCall (0001346743)                     True
Getting Betty Ann Lampman (0002013527)                    True
Getting Betty Ann Hardick (0002168819)                    True
Getting Betty Ann Cluthe (0002833711)                     True
Getting Betty Ann Ramseth (0003256558)                    True
Getting Anthony C. Deane (0001714107)                     True
Getting Anthony C. Price (0001805103)                     True
Getting Anthony C. Polson (0003823246)                    True
Getting C. Anthony (0001083525)                        

In [ ]:
from lib.allmusic import moveLocalFiles
moveLocalFiles()

In [ ]:
tmp = []
for line in tmp:
    if line.startswith("Getting"):
        artistID = line.split()[-2][1:-1]
        print(artistID)
        del searchedForErrors[artistID]

# Create Tabs Data

In [ ]:
def getTabs(rData):
    extraData = rData.profile.extra
    tabs = extraData.get('tabs', []) if isinstance(extraData,dict) else []
    retval = {tab.title: tab.href for tab in tabs}
    return retval

In [ ]:
mio   = allmusic.MusicDBIO()
tabsData = None
for modVal in range(100):
    modValTabsData = mio.data.getModValData(modVal).apply(getTabs)
    tabsData = concat([tabsData, modValTabsData]) if isinstance(tabsData,Series) else modValTabsData

# Download Artist Discography Data

# Parse

In [ ]:
from utils import PoolIO
pio = PoolIO("AllMusic")
pio.parse(force=True)
pio.meta()
pio.sum()
pio.search()